# Les 01 - Introductie tot AI-agenten

Welkom bij de eerste les in de cursus **AI-agenten voor beginners**!

Een **AI-agent** is een programma dat een groot taalmodel (LLM) gebruikt als zijn redeneermechanisme en *acties* kan ondernemen in de echte wereld — API’s aanroepen, databases bevragen of code uitvoeren — om een doel te bereiken namens een gebruiker.

In dit notitieboek bouw je je eerste agent: een **Reisbureau-Agent** die vakantiebestemmingen aanbeveelt. Onderweg leer je hoe je:

1. Verbinding maakt met Azure AI Foundry Agent Service met behulp van het **Microsoft Agent Framework**.
2. De agent een **tool** geeft — een gewone Python-functie die hij kan aanroepen.
3. De agent laat draaien en zijn reactie inspecteert.
4. De reactie van de agent token-voor-token streamt.


## Installatie

Voordat je deze notebook uitvoert, zorg ervoor dat je:

1. **Een Azure AI Foundry-project** hebt met een ingezet chatmodel (bijvoorbeeld `gpt-4o-mini`).
2. **Aangemeld bent bij de Azure CLI** — voer `az login` uit in je terminal.
3. **De benodigde omgevingsvariabelen hebt ingesteld:**
   - `AZURE_AI_PROJECT_ENDPOINT` — het eindpunt van je Azure AI Foundry-project.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — de naam van je ingezette model.

De onderstaande cel installeert de benodigde Python-pakketten.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Je Eerste Agent Maken

Een agent heeft twee dingen nodig:

- **Instructies** die vertellen *wie het is* en *hoe het zich moet gedragen* (een systeemprompt).
- **Tools** — Python-functies gedecoreerd met `@tool` die de agent kan aanroepen om informatie op te halen of acties uit te voeren.

Hieronder definiëren we een eenvoudige tool die een lijst met populaire vakantiebestemmingen retourneert. De agent zal deze tool gebruiken wanneer een gebruiker om reisaanbevelingen vraagt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Antwoorden

Voor een meer interactieve ervaring kun je de reactie van de agent **streamen**. In plaats van te wachten op de volledige reactie, levert de agent tekstfragmenten op zodra ze gegenereerd worden. Dit is vooral handig in chatinterfaces waar je de output in realtime wilt weergeven.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Samenvatting

In deze les heb je geleerd hoe je:

- **Een provider maakt** die verbinding maakt met Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Een tool definieert** met de `@tool` decorator zodat de agent jouw Python-functies kan aanroepen.
- **De agent uitvoert** met een gebruikersbericht en de reactie afdrukt.
- **Antwoorden streamt** voor realtime output.

In de volgende les zullen we agentische frameworks dieper verkennen en leren hoe je agenten krachtigere tools en multi-staps redeneer mogelijkheden kunt geven.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dit document is vertaald met behulp van de AI-vertalingsservice [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de originele taal moet als de gezaghebbende bron worden beschouwd. Voor cruciale informatie wordt een professionele menselijke vertaling aanbevolen. Wij aanvaarden geen aansprakelijkheid voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
